Setting Env Up

In [ ]:
import pandas as pd
import numpy as np
import gc

In [ ]:
df = pd.read_parquet("nfl.parquet")

In [ ]:
df.isna().sum()

,0
PlayId,0
GameId,0
Season,0
Week,0
Down,0
...,...
def11_PlayerHeight,0
def11_PlayerWeight,0
def11_PlayerAge,0
def11_Position,0


In [ ]:
df.head(5)

,PlayId,GameId,Season,Week,Down,Distance,HomeScoreBeforePlay,VisitorScoreBeforePlay,OffenseFormation,OffensePersonnel,...,def11_Dir_cos,def11_Ori_sin,def11_Ori_cos,def11_Sx,def11_Sy,def11_PlayerHeight,def11_PlayerWeight,def11_PlayerAge,def11_Position,Yards
0,1740578102,2.017091e+09,2017.0,1.0,3.0,2.0,0.0,0.0,5.0,9.0,...,0.997391,-0.954136,0.299374,0.099739,-0.007219,78.0,308.0,23.184120,5.0,8
1,1740578123,2.017091e+09,2017.0,1.0,1.0,10.0,0.0,0.0,5.0,9.0,...,-0.426727,-0.992799,-0.119790,-0.554745,-1.175695,78.0,308.0,23.184120,5.0,3
2,1740578173,2.017091e+09,2017.0,1.0,1.0,10.0,0.0,0.0,6.0,9.0,...,-0.140210,-0.999999,0.001222,-0.420630,2.970365,78.0,308.0,23.184120,5.0,5
3,1740578329,2.017091e+09,2017.0,1.0,2.0,2.0,0.0,0.0,3.0,53.0,...,-0.537300,-0.930673,0.365852,-1.343249,-2.108479,74.0,307.0,24.183435,5.0,2
4,1740578379,2.017091e+09,2017.0,1.0,1.0,10.0,7.0,0.0,5.0,18.0,...,-0.231239,-0.999585,-0.028794,-0.763087,-3.210560,74.0,320.0,23.597536,5.0,7


In [ ]:
print(list(df.columns))

['PlayId', 'GameId', 'Season', 'Week', 'Down', 'Distance', 'HomeScoreBeforePlay', 'VisitorScoreBeforePlay', 'OffenseFormation', 'OffensePersonnel', 'DefensePersonnel', 'DefendersInTheBox', 'PossessionTeam', 'FieldPosition', 'AbsYardLine', 'GameClockPct', 'rusher_X_rel', 'rusher_Y_rel', 'rusher_S', 'rusher_A', 'rusher_Dis', 'rusher_Dir_sin', 'rusher_Dir_cos', 'rusher_Ori_sin', 'rusher_Ori_cos', 'rusher_Sx', 'rusher_Sy', 'rusher_PlayerHeight', 'rusher_PlayerWeight', 'rusher_PlayerAge', 'rusher_Position', 'off1_X_rel', 'off1_Y_rel', 'off1_S', 'off1_A', 'off1_Dis', 'off1_Dir_sin', 'off1_Dir_cos', 'off1_Ori_sin', 'off1_Ori_cos', 'off1_Sx', 'off1_Sy', 'off1_PlayerHeight', 'off1_PlayerWeight', 'off1_PlayerAge', 'off1_Position', 'off2_X_rel', 'off2_Y_rel', 'off2_S', 'off2_A', 'off2_Dis', 'off2_Dir_sin', 'off2_Dir_cos', 'off2_Ori_sin', 'off2_Ori_cos', 'off2_Sx', 'off2_Sy', 'off2_PlayerHeight', 'off2_PlayerWeight', 'off2_PlayerAge', 'off2_Position', 'off3_X_rel', 'off3_Y_rel', 'off3_S', 'off3_A'

In [ ]:
#setting aside 10% of the dataset in case we need test data later
available_weeks = sorted(df['Week'].unique())
#reg season: weeks 1 to 17
#selecting the last 2 weeks for this
test_weeks = [16, 17]
df_test = df[df['Week'].isin(test_weeks)].copy()
df_train = df[~df['Week'].isin(test_weeks)].copy()
print(
    f"TRAIN dataset: {df_train.shape[0]} plays "
    f"(Weeks: {[w for w in available_weeks if w not in test_weeks]})")
print(
    f"TEST dataset:  {df_test.shape[0]} plays "
    f"(Weeks: {test_weeks})")
df_train.to_parquet(
    "nfl_train.parquet",
    index=False,
    compression="snappy")

df_test.to_parquet(
    "nfl_test.parquet",
    index=False,
    compression="snappy")

del df, df_train, df_test
gc.collect()

TRAIN dataset: 28086 plays (Weeks: [np.float32(1.0), np.float32(2.0), np.float32(3.0), np.float32(4.0), np.float32(5.0), np.float32(6.0), np.float32(7.0), np.float32(8.0), np.float32(9.0), np.float32(10.0), np.float32(11.0), np.float32(12.0), np.float32(13.0), np.float32(14.0), np.float32(15.0)])
TEST dataset:  2921 plays (Weeks: [16, 17])


0

###Baseline Models

Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import log_loss

In [ ]:
def evaluate(preds_prob_199, y_val):
    N = len(y_val)

    y_val_indices = y_val.astype(int) + 99
    loss_score = log_loss(y_val_indices, preds_prob_199, labels=np.arange(199))
    #convert raw probabilities into the cumulative distribution function (CDF)
    y_pred_cdf = np.cumsum(preds_prob_199, axis=1)
    y_pred_cdf = np.clip(y_pred_cdf, 0, 1)
    #build the True CDF
    y_true_cdf = np.zeros((N, 199))
    for i, yard_idx in enumerate(y_val_indices):
        y_true_cdf[i, yard_idx:] = 1
    #CRPS = mean squared error between the two CDFs
    crps_score = np.mean((y_pred_cdf - y_true_cdf) ** 2)
    return loss_score, crps_score

In [ ]:
df = pd.read_parquet("nfl_train.parquet")

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28086 entries, 0 to 28085
Columns: 347 entries, PlayId to Yards
dtypes: float32(345), int32(2)
memory usage: 37.2 MB


In [ ]:
#X, y, and grouping rule
columns_to_remove = ["PlayId", "GameId", "Yards", "Week", "Season"]
features = [col for col in df.columns if col not in columns_to_remove]
X = df[features].values
y = df["Yards"].values.astype(int)
week_groups = df["Week"].values
print(f"Training Structure: {X.shape[0]} plays and {X.shape[1]} feature columns.")

#configure GroupKFold with 5 folds, 3w/fold
gkf = GroupKFold(n_splits=5)
oof_predictions_prob = np.zeros((len(df), 199))
fold_scores_list = []
print("\nStarting Weekly Cross-Validation")
#fold Loop
for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups=week_groups)):
    print(f"\n> Fold {fold + 1} ")
    # Fold split
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[valid_idx], y[valid_idx]
    validation_weeks = np.unique(week_groups[valid_idx])
    print(f" validated on weeks: {validation_weeks}")
    #tree based ensemble
    rf_model = RandomForestClassifier(n_estimators=300, max_depth=None, max_features="sqrt", min_samples_leaf=14,
            random_state=48,  #Hutson
    )
    rf_model.fit(X_tr, y_tr)

    preds_prob = rf_model.predict_proba(X_va)
    preds_prob_199 = np.zeros((len(X_va), 199))
    for i, existing_class in enumerate(rf_model.classes_):
        official_index = (existing_class + 99)  #convert real yard value into 0-198 index
        preds_prob_199[:, official_index] = preds_prob[:, i]
    fold_loss, fold_crps = evaluate(preds_prob_199, y_va)
    oof_predictions_prob[valid_idx] = preds_prob_199
    fold_scores_list.append((fold_loss, fold_crps))
    print(f"Log Loss: {fold_loss:.4f} | CRPS: {fold_crps:.5f}")

    del X_tr, y_tr, X_va, y_va, rf_model
    gc.collect()

print("> FINAL OOF PERFORMANCE")
overall_loss, overall_crps = evaluate(oof_predictions_prob, y)
print(f"-> Overall Log Loss : {overall_loss:.4f}")
print(f"-> Overall CRPS     : {overall_crps:.5f}")

Training Structure: 28086 plays and 342 feature columns.

Starting Weekly Cross-Validation

> Fold 1 
 validated on weeks: [ 3. 10. 14.]
Log Loss: 2.8807 | CRPS: 0.01362

> Fold 2 
 validated on weeks: [ 2.  8. 11.]
Log Loss: 2.9259 | CRPS: 0.01341

> Fold 3 
 validated on weeks: [ 4.  6. 15.]
Log Loss: 2.9058 | CRPS: 0.01355

> Fold 4 
 validated on weeks: [ 1.  7. 13.]
Log Loss: 2.8596 | CRPS: 0.01323

> Fold 5 
 validated on weeks: [ 5.  9. 12.]
Log Loss: 2.9403 | CRPS: 0.01401
> FINAL OOF PERFORMANCE
-> Overall Log Loss : 2.9031
-> Overall CRPS     : 0.01357


$$- \ln\left(\frac{1}{199}\right) \approx 5.29$$

XGBoost

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import GroupKFold
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

In [ ]:
# X, y, and grouping rule
columns_to_remove = ["PlayId", "GameId", "Yards", "Week", "Season"]
features = [col for col in df.columns if col not in columns_to_remove]
X = df[features].values
y = df["Yards"].values.astype(int)
week_groups = df["Week"].values
print(f"Training Structure: {X.shape[0]} plays and {X.shape[1]} feature columns.")

#configure GroupKFold with 5 folds, 3w/fold
gkf = GroupKFold(n_splits=5)
oof_predictions_prob = np.zeros((len(df), 199))
fold_scores_list = []
print("\nStarting Weekly Cross-Validation")

#fold Loop
for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups=week_groups)):
    print(f"\n> Fold {fold + 1} ")
    #fold split
    X_tr, y_tr = X[train_idx], y[train_idx]
    X_va, y_va = X[valid_idx], y[valid_idx]
    validation_weeks = np.unique(week_groups[valid_idx])
    print(f" validated on weeks: {validation_weeks}")

    #encoder cause xgboost was complaining not all yards had frequencies > 0
    le = LabelEncoder()
    y_tr_encoded = le.fit_transform(y_tr)
    num_classes_found = len(le.classes_)

    #tree based ensemble
    xgb_model = XGBClassifier(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.05,
        objective="multi:softprob",
        num_class=num_classes_found,
        tree_method="hist",
        random_state=48,  # Hutson
    )
    xgb_model.fit(X_tr, y_tr_encoded)

    preds_prob = xgb_model.predict_proba(X_va)
    preds_prob_199 = np.zeros((len(X_va), 199))
    for i, class_value in enumerate(le.classes_):
        official_index = class_value + 99
        preds_prob_199[:, official_index] = preds_prob[:, i]

    fold_loss, fold_crps = evaluate(preds_prob_199, y_va)
    oof_predictions_prob[valid_idx] = preds_prob_199
    fold_scores_list.append((fold_loss, fold_crps))
    print(f"Log Loss: {fold_loss:.4f} | CRPS: {fold_crps:.5f}")
    del X_tr, y_tr, y_tr_encoded, X_va, y_va, xgb_model, le
    gc.collect()

print("\n> FINAL OOF PERFORMANCE")
overall_loss, overall_crps = evaluate(oof_predictions_prob, y)
print(f"-> Overall Log Loss : {overall_loss:.4f}")
print(f"-> Overall CRPS     : {overall_crps:.5f}")

Training Structure: 28086 plays and 342 feature columns.

Starting Weekly Cross-Validation

> Fold 1 
 validated on weeks: [ 3. 10. 14.]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Log Loss: 2.8355 | CRPS: 0.01346

> Fold 2 
 validated on weeks: [ 2.  8. 11.]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Log Loss: 2.8249 | CRPS: 0.01323

> Fold 3 
 validated on weeks: [ 4.  6. 15.]


...

Log Loss: 2.8222 | CRPS: 0.01338

> Fold 4 
 validated on weeks: [ 1.  7. 13.]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Log Loss: 2.8121 | CRPS: 0.01305

> Fold 5 
 validated on weeks: [ 5.  9. 12.]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:3001: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


Log Loss: 2.8752 | CRPS: 0.01387

> FINAL OOF PERFORMANCE
-> Overall Log Loss : 2.8343
-> Overall CRPS     : 0.01340


In [ ]:
# I also experimented with LGBM instead of XGBoost. However, I faced several problems I had to solve before making it work and at the end, the performance was kinda disappointing, so i didn't even include it as one of our "official" baselines. I kept de code cell here just for show

In [ ]:
# columns_to_remove = ["PlayId", "GameId", "Yards", "Week", "Season"]
# features = [col for col in df.columns if col not in columns_to_remove]
# X = df[features].values
# y = df["Yards"].values.astype(int)
# week_groups = df["Week"].values
# print(f"Training Structure: {X.shape[0]} plays and {X.shape[1]} feature columns.")

# gkf = GroupKFold(n_splits=5)
# oof_predictions_prob = np.zeros((len(df), 199))
# fold_scores_list = []
# print("\nStarting Weekly Cross-Validation (LightGBM)")

# for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups=week_groups)):
#     print(f"\n> Fold {fold + 1} ")
#     #separação dos dados
#     X_tr, y_tr = X[train_idx], y[train_idx]
#     X_va, y_va = X[valid_idx], y[valid_idx]
#     validation_weeks = np.unique(week_groups[valid_idx])
#     print(f" validated on weeks: {validation_weeks}")
#     le = LabelEncoder()
#     y_tr_encoded = le.fit_transform(y_tr)
#     num_classes_found = len(le.classes_)

#     #LightGBM
#     lgb_model = LGBMClassifier(
#     n_estimators=200,       
#     max_depth=8,          
#     num_leaves=63,
#     learning_rate=0.05,
#     min_child_samples=10,  
#     class_weight='balanced', 
#     random_state=48,
#     verbosity=-1,
#     )
#     lgb_model.fit(X_tr, y_tr_encoded)

#     preds_prob = lgb_model.predict_proba(X_va)
#     preds_prob_199 = np.zeros((len(X_va), 199))
#     for i, class_value in enumerate(le.classes_):
#         official_index = class_value + 99
#         preds_prob_199[:, official_index] = preds_prob[:, i]

#     fold_loss, fold_crps = evaluate(preds_prob_199, y_va)
#     oof_predictions_prob[valid_idx] = preds_prob_199
#     fold_scores_list.append((fold_loss, fold_crps))
#     print(f"Log Loss: {fold_loss:.4f} | CRPS: {fold_crps:.5f}")
#     del X_tr, y_tr, y_tr_encoded, X_va, y_va, lgb_model, le
#     gc.collect()
# overall_loss, overall_crps = evaluate(oof_predictions_prob, y)
# print(f"-> Overall Log Loss : {overall_loss:.4f}")
# print(f"-> Overall CRPS     : {overall_crps:.5f}")

Training Structure: 28086 plays and 342 feature columns.

Starting Weekly Cross-Validation (LightGBM)

> Fold 1 
 validated on weeks: [ 3. 10. 14.]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


y_val range: -10 to 99
y_val_indices range: 89 to 198
Log Loss: 3.5943 | CRPS: 0.01498

> Fold 2 
 validated on weeks: [ 2.  8. 11.]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


y_val range: -12 to 87
y_val_indices range: 87 to 186
Log Loss: 3.5525 | CRPS: 0.01469

> Fold 3 
 validated on weeks: [ 4.  6. 15.]


...

y_val range: -14 to 88
y_val_indices range: 85 to 187
Log Loss: 3.5953 | CRPS: 0.01491

> Fold 4 
 validated on weeks: [ 1.  7. 13.]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


y_val range: -11 to 90
y_val_indices range: 88 to 189
Log Loss: 3.6281 | CRPS: 0.01462

> Fold 5 
 validated on weeks: [ 5.  9. 12.]


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


y_val range: -15 to 97
y_val_indices range: 84 to 196
Log Loss: 3.6357 | CRPS: 0.01541

> FINAL OOF PERFORMANCE (LightGBM)
y_val range: -15 to 99
y_val_indices range: 84 to 198
-> Overall Log Loss : 3.6010
-> Overall CRPS     : 0.01492


In [ ]:
# I previouslt thought XGBoost and LightGBM were pretty similar algorithms, so I expected them to be close in performance. Since that's not what happened, I looked for reasons why that'd be the case. And the explanation I learned was:

# *"LightGBM grows trees leaf-wise (by the leaf with the highest gain), which is more aggressive and quickly specializes in dominant classes."* 
# With +100 unbalanced classes, it learns to confidently kick the most common class, and softmax amplifies this, resulting mostly in probability 1.0 in one class, 0.0 in all others.

NGBoost

In [ ]:
from ngboost import NGBRegressor
from ngboost.distns import Normal
from scipy.stats import norm
import numpy as np
import gc

# X, y
columns_to_remove = ["PlayId", "GameId", "Yards", "Week", "Season"]
features = [col for col in df.columns if col not in columns_to_remove]
X = df[features].values
y = df["Yards"].values.astype(float)
week_groups = df["Week"].values

#time based split
last_weeks = np.sort(np.unique(week_groups))[-3:]
val_mask = np.isin(week_groups, last_weeks)
train_mask = ~val_mask

X_tr, y_tr = X[train_mask], y[train_mask]
X_va, y_va = X[val_mask], y[val_mask]
print(f"Train: {X_tr.shape[0]} plays | Val: {X_va.shape[0]} plays (weeks {last_weeks})")

ngb_model = NGBRegressor(
    Dist=Normal,
    n_estimators=0,
    learning_rate=0.01,
    random_state=75, #Dobes
    verbose=True,)
ngb_model.fit(X_tr, y_tr)

pred_dist = ngb_model.pred_dist(X_va)
mu = pred_dist.params["loc"]
sigma = pred_dist.params["scale"]

# CDF avia cumsum of bins
yards = np.arange(-99, 100)
preds_cdf_199 = norm.cdf(yards[np.newaxis, :], loc=mu[:, np.newaxis], scale=sigma[:, np.newaxis])
preds_prob_199 = np.diff(preds_cdf_199, prepend=0, axis=1)
preds_prob_199 = np.clip(preds_prob_199, 0, 1)

loss, crps = evaluate(preds_prob_199, y_va)
print(f"\n> NGBoost Normal baseline")
print(f"-> Log Loss : {loss:.4f}")
print(f"-> CRPS     : {crps:.5f}")

Train: 23688 plays | Val: 4398 plays (weeks [13. 14. 15.])
[iter 0] loss=3.2888 val_loss=0.0000 scale=1.0000 norm=4.0139
[iter 100] loss=3.1124 val_loss=0.0000 scale=2.0000 norm=7.6149
y_val range: -14.0 to 99.0
y_val_indices range: 85 to 198

> NGBoost Normal baseline
-> Log Loss : 3.1566
-> CRPS     : 0.01444


MLP

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [ ]:
#feature split
play_cols = ['GameId', 'PlayId', 'Yards', 'Week', 'Season']
feature_cols = [c for c in df.columns if c not in play_cols]
X = df[feature_cols].values.astype(np.float32)
y = df['Yards'].values.astype(int)
week_groups = df['Week'].values
print(f"X shape: {X.shape} | y range: {y.min()} to {y.max()}")

X shape: (31006, 660) | y range: -15 to 99


In [ ]:
X = df[feature_cols].values.astype(np.float32)
X = np.nan_to_num(X, nan=0.0)

In [ ]:
#time based train/test split
last_weeks = np.sort(np.unique(week_groups))[-3:]
val_mask = np.isin(week_groups, last_weeks)
X_tr, y_tr = X[~val_mask], y[~val_mask]
X_va, y_va = X[val_mask],  y[val_mask]
print(f"Train: {len(X_tr)} plays | Val: {len(X_va)} plays (weeks {last_weeks})")


Train: 26612 plays | Val: 4394 plays (weeks [15 16 17])


In [ ]:
scaler = StandardScaler()
X_tr = scaler.fit_transform(X_tr)
X_va = scaler.transform(X_va)

In [ ]:
class PlayDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        #y 0-198 for CrossEntropyLoss
        self.y = torch.tensor(y + 99, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(PlayDataset(X_tr, y_tr), batch_size=256, shuffle=True)
val_loader = DataLoader(PlayDataset(X_va, y_va), batch_size=256, shuffle=False)


In [ ]:
#simple architecture tbw
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim=199):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, output_dim),)

    def forward(self, x):
        return self.net(x)  #softmax


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = MLP(input_dim=X_tr.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

In [ ]:
#and train
N_EPOCHS = 60
best_crps = np.inf
best_state = None

for epoch in range(N_EPOCHS):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(X_batch)
    train_loss /= len(X_tr)

    model.eval()
    all_probs = []
    with torch.no_grad():
        for X_batch, _ in val_loader:
            logits = model(X_batch.to(device))
            probs  = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.append(probs)

    preds_prob_199 = np.vstack(all_probs)  #(n_val, 199)
    val_loss, val_crps = evaluate(preds_prob_199, y_va)
    scheduler.step(val_crps)

    if val_crps < best_crps:
        best_crps  = val_crps
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:02d}/{N_EPOCHS} | "
              f"Train Loss: {train_loss:.4f} | "
              f"Val Log Loss: {val_loss:.4f} | "
              f"Val CRPS: {val_crps:.5f}")

Epoch 05/60 | Train Loss: 2.7521 | Val Log Loss: 2.8379 | Val CRPS: 0.01308 ✓
Epoch 10/60 | Train Loss: 2.7433 | Val Log Loss: 2.8353 | Val CRPS: 0.01307
Epoch 15/60 | Train Loss: 2.7315 | Val Log Loss: 2.8323 | Val CRPS: 0.01304 ✓
Epoch 20/60 | Train Loss: 2.7246 | Val Log Loss: 2.8300 | Val CRPS: 0.01302 ✓
Epoch 25/60 | Train Loss: 2.7173 | Val Log Loss: 2.8287 | Val CRPS: 0.01301 ✓
Epoch 30/60 | Train Loss: 2.7079 | Val Log Loss: 2.8269 | Val CRPS: 0.01300 ✓
Epoch 35/60 | Train Loss: 2.7039 | Val Log Loss: 2.8264 | Val CRPS: 0.01299 ✓
Epoch 40/60 | Train Loss: 2.6959 | Val Log Loss: 2.8256 | Val CRPS: 0.01298 ✓
Epoch 45/60 | Train Loss: 2.6925 | Val Log Loss: 2.8251 | Val CRPS: 0.01297
Epoch 50/60 | Train Loss: 2.6838 | Val Log Loss: 2.8245 | Val CRPS: 0.01297 ✓
Epoch 55/60 | Train Loss: 2.6806 | Val Log Loss: 2.8240 | Val CRPS: 0.01296
Epoch 60/60 | Train Loss: 2.6745 | Val Log Loss: 2.8249 | Val CRPS: 0.01296


In [ ]:
model.load_state_dict(best_state)
model.eval()
all_probs = []
with torch.no_grad():
    for X_batch, _ in val_loader:
        logits = model(X_batch.to(device))
        probs  = torch.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)

preds_prob_199 = np.vstack(all_probs)
final_loss, final_crps = evaluate(preds_prob_199, y_va)
print(f"\n> MLP")
print(f"-> Log Loss : {final_loss:.4f}")
print(f"-> CRPS     : {final_crps:.5f}")


> MLP
-> Log Loss : 2.8242
-> CRPS     : 0.01295
